In [1]:
# Import relevant libraries.
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from datetime import datetime
import re

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('words')
nltk.download('omw-1.4')
from nltk.corpus import stopwords
from nltk.corpus import words
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package words to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
# Load dataset. Change directory as required.
df = pd.read_csv('euro_speeches.csv')

In [3]:
df.head()

,reference,country,date,title,author,is_gov,text
0,r970207a_ECB,euro area,07/02/1997,Conference organised by the Hungarian Banking ...,lamfalussy,0,"For at least three reasons, I have accepted wi..."
1,r970310a_ECB,euro area,10/03/1997,Securing the benefits of EMU,lamfalussy,0,It is a great pleasure to be with you today he...
2,r970422a_ECB,euro area,22/04/1997,Convergence and the role of the European Centr...,lamfalussy,0,These remarks will touch on the following topi...
3,r970430a_ECB,euro area,30/04/1997,The operation of monetary policy in stage thre...,lamfalussy,0,"I am delighted to be here today in New York, i..."
4,r970513a_ECB,euro area,13/05/1997,The European Central Bank: independent and acc...,lamfalussy,0,Against a background of both historical experi...


In [4]:
df.country.value_counts()

euro area    2478
Name: country, dtype: int64

In [5]:
# Add a column to calculate the string length per speech.
df['len'] = df['text'].str.len()
df

,reference,country,date,title,author,is_gov,text,len
0,r970207a_ECB,euro area,07/02/1997,Conference organised by the Hungarian Banking ...,lamfalussy,0,"For at least three reasons, I have accepted wi...",22216
1,r970310a_ECB,euro area,10/03/1997,Securing the benefits of EMU,lamfalussy,0,It is a great pleasure to be with you today he...,22112
2,r970422a_ECB,euro area,22/04/1997,Convergence and the role of the European Centr...,lamfalussy,0,These remarks will touch on the following topi...,16025
3,r970430a_ECB,euro area,30/04/1997,The operation of monetary policy in stage thre...,lamfalussy,0,"I am delighted to be here today in New York, i...",23073
4,r970513a_ECB,euro area,13/05/1997,The European Central Bank: independent and acc...,lamfalussy,0,Against a background of both historical experi...,17384
...,...,...,...,...,...,...,...,...
2473,r221014a_ECB,euro area,14/10/2022,IMFC Statement,lagarde,1,Global growth momentum has slowed since our pr...,11192
2474,r221031a_ECB,euro area,31/10/2022,NO_INFO,lane,0,Notes: The standard deviation of monthly perce...,5729
2475,r221103a_ECB,euro area,03/11/2022,Mind the step: calibrating monetary policy in ...,panetta,0,The euro area is facing a sequence of unpreced...,24075
2476,r221104b_ECB,euro area,04/11/2022,The euro area economy and the energy transition,guindos,0,I am very pleased to be taking part in this ev...,10989


In [6]:
# Text cleaning (Convert to lower case and remove punctuation)
df['text'] = df['text'].str.lower().str.replace('[^\w\s]', '', regex=True)

In [7]:
# VADER sentiment (Calculate Sentiment intensity analysis using Vadar sentiment)
sia = SentimentIntensityAnalyzer()
df[['neg', 'neu', 'pos', 'compound']] = df['text'].apply(lambda x: pd.Series(sia.polarity_scores(x)))

In [8]:
# TextBlob sentiment (Calculate polarity and subjectivity using TextBlob)
df[['polarity','subjectivity']] = df['text'].apply(lambda x: pd.Series(TextBlob(x).sentiment))

In [9]:
# Load Loughran–McDonald Dictionary
lm_dict = pd.read_csv("LM_dictionary.csv")  
print("LM Columns:", lm_dict.columns)  # check columns

LM Columns: Index(['Word', 'Negative', 'Positive', 'Uncertainty', 'Litigious', 'Strong',
       'Weak', 'Constraining'],
      dtype='object')


In [10]:
# Create a mapping: Word -> list of categories
lm_dict_map = {}
for _, row in lm_dict.iterrows():
    word = row['Word'].upper()
    categories = [col for col in lm_dict.columns[1:] if row[col] > 0]  # skip 'Word' column
    lm_dict_map[word] = categories

In [11]:
# Function to compute LM sentiment
def lm_sentiment(text, lm_dict_map):
    words = re.findall(r'\b\w+\b', text.upper())
    pos = sum(1 for w in words if 'Positive' in lm_dict_map.get(w, []))
    neg = sum(1 for w in words if 'Negative' in lm_dict_map.get(w, []))
    total = pos + neg
    return 0 if total == 0 else (pos - neg)/total

In [12]:
#import re
# Apply LM sentiment
df['lm_score'] = df['text'].apply(lambda x: lm_sentiment(x, lm_dict_map))

In [13]:
# Combined Score (simple average)
df['combined_score'] = (df['compound'] + df['polarity'] + df['lm_score']) / 3

In [14]:
# LM Label Thresholds
def lm_label(score, pos_thresh=0.05, neg_thresh=-0.05):
    """
    Assigns a sentiment label based on LM score thresholds.
    
    Parameters:
        score: LM sentiment score ([-1,1])
        pos_thresh: threshold above which text is Positive
        neg_thresh: threshold below which text is Negative
        
    Returns:
        'Positive', 'Negative', or 'Neutral'
    """
    if score > pos_thresh:
        return "Positive"
    elif score < neg_thresh:
        return "Negative"
    else:
        return "Neutral"
    
df['lm_label'] = df['lm_score'].apply(lambda x: lm_label(x))    

# Export Selected Columns
columns_to_export = [
    'reference','date','text', 
    'neg', 'neu', 'pos', 'compound', 
    'polarity', 'subjectivity', 
    'lm_score', 
    'combined_score', 
    'lm_label'
]

In [15]:
# Export CSV
df.to_csv("euro_sentiment_labeled.csv", columns=columns_to_export, index=False, encoding='utf-8')

In [16]:
# Export Excel
df.to_excel("Euro_sentiment_labeled.xlsx", columns=columns_to_export, index=False, sheet_name="Euro_Sentiment")